In [95]:
!pip install -U pip transformers

In [96]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

In [97]:
checkpoint = 'facebook/nllb-200-distilled-600M'
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

In [98]:
print(f"{len(tokenizer.vocab)}\n")

tokenizer.vocab

256204



{'بیح': 210484,
 '▁དང': 165217,
 '▁edil': 25778,
 '▁биринчи': 82617,
 '▁dije': 63488,
 'र्म': 9594,
 '▁շ': 8515,
 'okur': 74065,
 '▁porządku': 116030,
 'osora': 149580,
 '▁vakulu': 113197,
 'ghanaghan': 214314,
 '▁qə': 18775,
 'mik': 38044,
 '▁şäher': 108647,
 '▁hran': 80626,
 '▁избори': 146674,
 '▁पढ़ें': 166583,
 '▁quieren': 153151,
 '▁프로': 12018,
 '▁Jason': 61083,
 'вување': 201993,
 '▁올해': 58469,
 '▁従': 121168,
 'ให้เธอ': 241180,
 'ետ': 4568,
 '▁munat': 207937,
 '▁rived': 242821,
 '▁ostale': 189175,
 '▁vásár': 114105,
 'ാണെന്നു': 245829,
 'ព្រ': 41791,
 'ናዊ': 116192,
 '▁تحویل': 219799,
 'éo': 84656,
 '▁үргэл': 86849,
 '▁airson': 12080,
 'zhak': 109914,
 '▁prof': 3445,
 'angeni': 142045,
 '▁چۈش': 32860,
 '▁teatr': 211102,
 'astan': 89621,
 '▁వి': 5105,
 'Мин': 120203,
 '▁Дон': 96061,
 '▁affected': 192060,
 '▁ŵaku': 43500,
 '▁salah': 9292,
 '抗议': 228115,
 '▁bad': 6892,
 '▁dias': 50245,
 '▁raand': 147230,
 '▁ଅବ': 37951,
 'チョ': 117185,
 '▁ыйык': 71590,
 'rater': 206137,
 '▁author': 324

In [99]:
thai_char_min = 0x0E00
thai_char_max = 0x0E7F

thai_tokens = [
    token for token in tokenizer.vocab.keys()
    if any(thai_char_min <= ord(char) <= thai_char_max for char in token)
]

thai_token_count = len(thai_tokens)
sample_size = 20
thai_tokens_sample = thai_tokens[:sample_size]


print(f"{thai_token_count}\n")
for token in thai_tokens_sample:
  print(token)

1712

หมด
ื่
ทิ
็บ
วันนี้
▁ฉันไม่
พล
▁ตอนนี้
▁ป
ูล
ซี่
ปฏิ
รร
ราย
▁นี่คือ
พเจ้า
นั่น
ฆ่า
เพ
็น


In [100]:
import tensorflow as tf
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import numpy as np
import math

In [101]:
sentence = 'Work hard, play harder'

In [102]:
cleaned_sentence = sentence.replace(',', '')
cleaned_sentence

'Work hard play harder'

In [103]:
words = cleaned_sentence.split()
words

['Work', 'hard', 'play', 'harder']

In [104]:
sorted_words = sorted(words)
sorted_words

['Work', 'hard', 'harder', 'play']

In [105]:
dc = {word: index for index, word in enumerate(sorted_words)}
dc

{'Work': 0, 'hard': 1, 'harder': 2, 'play': 3}

In [106]:
sentence_int = tf.constant(
    [dc[s] for s in sentence.replace(',', '').split()],
    dtype=tf.int32
)

In [107]:
print(sentence)
print(sentence_int)

Work hard, play harder
tf.Tensor([0 1 3 2], shape=(4,), dtype=int32)


In [108]:
# สร้าง embedding layer
tf.random.set_seed(123)
vocab_size = 50_000
embedding_dim = 2

embed = tf.keras.layers.Embedding(input_dim=vocab_size, output_dim=embedding_dim)

In [109]:
embedded_sentence = embed(sentence_int)

In [110]:
embedded_sentence

<tf.Tensor: shape=(4, 2), dtype=float32, numpy=
array([[ 0.0273474 ,  0.02690465],
       [ 0.02351924, -0.01599505],
       [ 0.02245296,  0.0379287 ],
       [ 0.0418286 ,  0.0237045 ]], dtype=float32)>

In [111]:
tf.random.set_seed(123)
vocab_size = 50_000
embedding_dim = 2

dummy_input = tf.constant([0, 1, 2], dtype=tf.int32)

# Case 1 Default initializer (RandomUniform(-0.05, 0.05))
embed_default = tf.keras.layers.Embedding(input_dim=vocab_size, output_dim=embedding_dim)
_ = embed_default(dummy_input) # เรียกใช้งาน layer เพื่อสร้าง weights
weights_default = embed_default.get_weights()[0].flatten()
weights_default.shape

(100000,)

In [112]:
# Case 2 GlorotUniform initializer
tf.random.set_seed(123)
embed_glorot = tf.keras.layers.Embedding(
    input_dim=vocab_size,
    output_dim=embedding_dim,
    embeddings_initializer=tf.keras.initializers.GlorotUniform()
)
_ = embed_glorot(dummy_input) # เรียกใช้งาน layer เพื่อสร้าง weights
weights_glorot = embed_glorot.get_weights()[0].flatten()
weights_glorot.shape

(100000,)

In [113]:
fig = make_subplots(rows=1, cols=1)

fig.add_trace(go.Histogram(x=weights_default, nbinsx=50, name="Default Uniform [-0.05, 0.05]", opacity=0.6))
fig.add_trace(go.Histogram(x=weights_glorot, nbinsx=50, name="Glorot Uniform", opacity=0.6))

fig.update_layout(
    title_text='Embedding Layer Initialization Comparison',
    xaxis_title_text='Weight values',
    yaxis_title_text='Frequency',
    barmode='overlay',
    legend_orientation="h",
    legend_yanchor="bottom",
    legend_y=1.02,
    legend_xanchor="right",
    legend_x=1
)

fig.show()

print("Default initializer range ", weights_default.min(), weights_default.max())
print("Glorot initializer range ", weights_glorot.min(), weights_glorot.max())

Default initializer range  -0.049999535 0.04999863
Glorot initializer range  -0.0109542245 0.010954059


In [114]:
def glorot_uniform_limits(fan_in, fan_out):
    limit = math.sqrt(6.0 / (fan_in + fan_out))
    a, b = -limit, limit
    return a, b

# ตัวอย่าง Embedding layer (vocab_size=50000, embedding_dim=2)
fan_in = 50000
fan_out = 2

a, b = glorot_uniform_limits(fan_in, fan_out)
print("Glorot Uniform a =", a)
print("Glorot Uniform b =", b)

Glorot Uniform a = -0.010954232067652772
Glorot Uniform b = 0.010954232067652772


In [115]:
model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint)

In [116]:
token_embedding_layer = model.model.encoder.embed_tokens
token_embedding_layer.weight.shape

torch.Size([256206, 1024])

In [117]:
long_sentence = "In the vast realm of natural language processing, understanding the nuances of how models handle sequential data is crucial. Positional encoding plays a vital role in providing this essential information to the model, allowing it to differentiate between words at different positions in a sentence, which is fundamental for tasks like translation, summarization, and text generation."

In [118]:
tokens = tokenizer(long_sentence, return_tensors="pt")

print(tokens['input_ids'][0])


tensor([256047,    717,    349,  14430,  12284, 248070,    452,  25307,  65445,
        157278, 248079, 133930,    349,    713,  75831,    452,  11657, 141057,
         47274, 116914, 124785,   6067,    248, 182071, 248075,  12013,  58409,
         12025, 246156,   3054,    705,      9, 104781,  76065,    108, 174693,
          3423, 140515,  18781,    202,    349,  14916, 248079,  82935,     87,
           796,    202,  53054,    502,  25914,  51744,    230,  30158, 199073,
           108,      9, 109267, 248079,   9089,    248,  75529,    351, 226047,
          6399, 200356, 248079,   2493, 109207, 181953, 248079,    540,  35883,
        120531, 248075,      2])


In [119]:
len(tokens['input_ids'][0])

75

In [120]:
token_embedding_layer(tokens['input_ids'][0][0]).shape

torch.Size([1024])

In [121]:
token_embeddings = token_embedding_layer(tokens['input_ids'][0])

print("Token Embedding Matrix shape", token_embeddings.shape)
token_embeddings

Token Embedding Matrix shape torch.Size([75, 1024])


tensor([[-5.0000e+00, -1.2725e+00, -9.3604e-01,  ..., -1.8297e+01,
         -9.1328e+00, -1.0672e+01],
        [ 2.6416e-01,  2.6831e-01,  2.0117e-01,  ...,  3.2715e+00,
         -3.2402e+00,  3.1738e+00],
        [ 4.3579e-01, -2.3352e-01,  2.6825e-02,  ...,  5.4648e+00,
          2.7129e+00,  5.5430e+00],
        ...,
        [ 8.5859e+00, -4.5391e+00, -4.7314e-01,  ..., -7.9529e-02,
          7.4844e+00, -7.5156e+00],
        [-2.4863e+00, -2.7515e-01,  5.6114e-03,  ...,  1.0180e+01,
         -7.2422e+00, -4.8047e+00],
        [-7.8320e-01, -9.0527e-01, -9.4482e-01,  ...,  3.1078e+01,
         -8.1494e-01, -8.7354e-01]], grad_fn=<MulBackward0>)

In [122]:
import plotly.express as px

token_embeddings_np = token_embeddings.detach().numpy()

fig = px.imshow(
    token_embeddings_np,
    color_continuous_scale="RdBu",
    labels=dict(x="Embedding Dimension", y="Token Index", color="Value"),
    title="Token Embedding Heatmap"
)

fig.update_xaxes(side="top")
fig.update_layout(height=500, width=900)
fig.show()

In [123]:
d = embedded_sentence.shape[-1]
d

2

In [124]:
d_q, d_k, d_v = 2, 2, 4

d_q, d_k, d_v

(2, 2, 4)

In [125]:
tf.random.set_seed(123)
W_query = tf.Variable(tf.random.uniform((d, d_q)), trainable=True)
W_key   = tf.Variable(tf.random.uniform((d, d_k)), trainable=True)
W_value = tf.Variable(tf.random.uniform((d, d_v)), trainable=True)

In [126]:
print(W_query.shape, W_key.shape, W_value.shape)

(2, 2) (2, 2) (2, 4)


In [127]:
W_query

<tf.Variable 'Variable:0' shape=(2, 2) dtype=float32, numpy=
array([[0.12615311, 0.5727513 ],
       [0.2993133 , 0.5461836 ]], dtype=float32)>

In [128]:
W_key

<tf.Variable 'Variable:0' shape=(2, 2) dtype=float32, numpy=
array([[0.88968754, 0.12354946],
       [0.7718717 , 0.6850728 ]], dtype=float32)>

In [129]:
W_value

<tf.Variable 'Variable:0' shape=(2, 4) dtype=float32, numpy=
array([[0.48962688, 0.5857923 , 0.36451697, 0.6550509 ],
       [0.9075084 , 0.37557673, 0.6882372 , 0.25384045]], dtype=float32)>

In [130]:
embedded_sentence

<tf.Tensor: shape=(4, 2), dtype=float32, numpy=
array([[ 0.0273474 ,  0.02690465],
       [ 0.02351924, -0.01599505],
       [ 0.02245296,  0.0379287 ],
       [ 0.0418286 ,  0.0237045 ]], dtype=float32)>

In [131]:
queries = tf.matmul(embedded_sentence, W_query)
keys    = tf.matmul(embedded_sentence, W_key)
values  = tf.matmul(embedded_sentence, W_value)

In [132]:
print("Queries shape", queries.shape)
queries

Queries shape (4, 2)


<tf.Tensor: shape=(4, 2), dtype=float32, numpy=
array([[ 0.01150288,  0.03035814],
       [-0.0018205 ,  0.00473445],
       [ 0.01418508,  0.033576  ],
       [ 0.01237188,  0.03690439]], dtype=float32)>

In [133]:
print("Keys shape", keys.shape)
keys

Keys shape (4, 2)


<tf.Tensor: shape=(4, 2), dtype=float32, numpy=
array([[ 0.04509758,  0.0218104 ],
       [ 0.00857865, -0.00805198],
       [ 0.04925221,  0.02875797],
       [ 0.05551122,  0.02140721]], dtype=float32)>

In [134]:
print("Values shape", values.shape)
values

Values shape (4, 4)


<tf.Tensor: shape=(4, 4), dtype=float32, numpy=
array([[ 0.03780622,  0.02612466,  0.02848537,  0.02474342],
       [-0.00299999,  0.00777002, -0.00243522,  0.01134611],
       [ 0.04541419,  0.02739791,  0.03428843,  0.02433567],
       [ 0.04199244,  0.03340573,  0.03156155,  0.03341702]],
      dtype=float32)>

In [135]:
omega = tf.matmul(queries, keys, transpose_b=True)

In [136]:
print("Omega shape", omega.shape)
print("Omega (Unnormalized attention weights)")
print(omega)

Omega shape (4, 4)
Omega (Unnormalized attention weights)
tf.Tensor(
[[ 1.1808753e-03 -1.4576394e-04  1.4395809e-03  1.2884219e-03]
 [ 2.1159802e-05 -5.3739153e-05  4.6489164e-05  2.9282819e-07]
 [ 1.3720186e-03 -1.4866446e-04  1.6642241e-03  1.5061994e-03]
 [ 1.3628416e-03 -1.9101943e-04  1.6706381e-03  1.4767983e-03]], shape=(4, 4), dtype=float32)


In [137]:
d_k = tf.cast(d_k, tf.float32)

scaled_omega = omega / tf.sqrt(d_k)

attention_weights = tf.nn.softmax(scaled_omega, axis=-1)

print("Attention Weights")
print(attention_weights)

Attention Weights
tf.Tensor(
[[0.25004244 0.24980795 0.25008816 0.25006142]
 [0.25000313 0.24998987 0.2500076  0.24999942]
 [0.25004834 0.24977961 0.25010002 0.25007206]
 [0.25005    0.24977542 0.25010443 0.25007015]], shape=(4, 4), dtype=float32)


In [138]:
row_sums = tf.reduce_sum(attention_weights, axis=-1)

print("Sum of each row in attention_weights")
row_sums

Sum of each row in attention_weights


<tf.Tensor: shape=(4,), dtype=float32, numpy=array([1., 1., 1., 1.], dtype=float32)>

In [139]:
context_vector = tf.matmul(attention_weights, values)

print("Context Vector shape", context_vector.shape)
print(context_vector)

Context Vector shape (4, 4)
tf.Tensor(
[[0.03056198 0.02367866 0.02298167 0.02346363]
 [0.03055369 0.02367477 0.02297539 0.02346069]
 [0.03056327 0.02367928 0.02298265 0.02346409]
 [0.03056347 0.02367935 0.0229828  0.02346413]], shape=(4, 4), dtype=float32)


In [140]:
class SelfAttention(tf.keras.layers.Layer):
    def __init__(self, d_in, d_out_kq, d_out_v):
        super().__init__()
        self.d_out_kq = d_out_kq

        self.W_query = tf.Variable(
            tf.random.uniform((d_in, d_out_kq)), trainable=True
        )
        self.W_key = tf.Variable(
            tf.random.uniform((d_in, d_out_kq)), trainable=True
        )
        self.W_value = tf.Variable(
            tf.random.uniform((d_in, d_out_v)), trainable=True
        )

    def call(self, x):
        keys = tf.matmul(x, self.W_key)      # [T, d_out_kq]
        queries = tf.matmul(x, self.W_query) # [T, d_out_kq]
        values = tf.matmul(x, self.W_value)  # [T, d_out_v]

        # Attention scores: QKᵀ
        attn_scores = tf.matmul(queries, keys, transpose_b=True)  # [T, T]

        # Softmax (scaled by sqrt(d_k))
        attn_weights = tf.nn.softmax(
            attn_scores / tf.math.sqrt(tf.cast(self.d_out_kq, tf.float32)), axis=-1
        )  # [T, T]

        # Weighted sum
        context_vec = tf.matmul(attn_weights, values)  # [T, d_out_v]
        return context_vec

In [141]:
embedded_sentence

<tf.Tensor: shape=(4, 2), dtype=float32, numpy=
array([[ 0.0273474 ,  0.02690465],
       [ 0.02351924, -0.01599505],
       [ 0.02245296,  0.0379287 ],
       [ 0.0418286 ,  0.0237045 ]], dtype=float32)>

In [142]:
tf.random.set_seed(123)

d_in, d_out_kq, d_out_v = 2, 2, 4

sa = SelfAttention(d_in, d_out_kq, d_out_v)

out = sa(embedded_sentence)

print(out.shape)  # (T, d_out_v)
print(out.numpy())

(4, 4)
[[0.03056198 0.02367866 0.02298167 0.02346363]
 [0.03055369 0.02367477 0.02297539 0.02346069]
 [0.03056327 0.02367928 0.02298265 0.02346409]
 [0.03056347 0.02367935 0.0229828  0.02346413]]


In [143]:
class MultiHeadAttentionWrapper(tf.keras.layers.Layer):
    def __init__(self, d_in, d_out_kq, d_out_v, num_heads):
        super().__init__()
        self.heads = [
            SelfAttention(d_in, d_out_kq, d_out_v)
            for _ in range(num_heads)
        ]

    def call(self, x):
        # รันทุก head แล้ว concat ตามแกนสุดท้าย
        head_outputs = [head(x) for head in self.heads]   # list of [T, d_out_v]
        return tf.concat(head_outputs, axis=-1)           # [T, num_heads * d_out_v]

In [144]:
tf.random.set_seed(123)

d_in, d_out_kq, d_out_v = 2, 2, 1

sa = SelfAttention(d_in, d_out_kq, d_out_v)

# ถ้า embedded_sentence.shape = [T, d_in] เช่น [6, 3]
out = sa(embedded_sentence)

print(out.shape)   # (T, d_out_v) -> (6, 1)
print(out.numpy())

(4, 1)
[[0.02472453]
 [0.02471897]
 [0.0247254 ]
 [0.02472552]]


In [145]:
tf.random.set_seed(123)

block_size = embedded_sentence.shape[0]   # [T, d_in] → T = sequence length

mha = MultiHeadAttentionWrapper(
    d_in, d_out_kq, d_out_v, num_heads=3
)

# run MHA
context_vecs = mha(embedded_sentence)   # [T, num_heads * d_out_v]

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tf.Tensor(
[[0.02472453 0.02903921 0.03787982]
 [0.02471897 0.02903681 0.03787687]
 [0.0247254  0.02903946 0.03787924]
 [0.02472552 0.02903995 0.03788297]], shape=(4, 3), dtype=float32)
context_vecs.shape: (4, 3)
